In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.model_selection import train_test_split
import tracemalloc
from rasterio.enums import Compression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
import sklearn
import joblib
from tqdm import tqdm
import time

from skopt import BayesSearchCV
from skopt.space import Real, Integer

import rasterio
from quantile_forest import RandomForestQuantileRegressor

ModuleNotFoundError: No module named 'quantile_forest'

In [3]:
###########################################################################
#######################                            ########################
#######################     tiff reorder part 01   ########################
#######################                            ########################
###########################################################################


# tiff band reorder
# Load the CSV file containing band selection info
csv_file = "./predictionMap2025Jan/predictionBandsInfo.csv"  # 
df = pd.read_csv(csv_file)

# Read the TIFF file
tiff_file = "/blue/changzhao/ji.song/Features_Stack_FL/composite_all.tif" # 110 band
# Save the new TIFF file
output_tiff = "/blue/changzhao/zhou.tang/SOC/InputBands_NearFire.tif"

# Prepare a list to store the selected bands
selected_bands = []

print("===start reorder bands==")
i = 0
for _, row in df.iterrows():
    print(f"start process the band {_}")
    band_indices = str(row["tiff_order"]).split(",")  # Handle multiple bands
    band_indices = [int(idx) for idx in band_indices]  # Convert to zero-based index
    if len(band_indices) >1:
        with rasterio.open(tiff_file) as src:
            print(src.nodatavals)
            meta = src.meta.copy()  # Save metadata
            selected_band = src.read(band_indices)  # Load all bands as a NumPy array (shape: bands, height, width)

        selected_band = np.sum(selected_band, axis=0)        
        selected_bands.append(selected_band)
        print(f"finish the band {row['Feature_name']} with index ")
        i = i + 1

print(f"total bands is {i}")

# Update metadata for the new TIFF
new_tiff_array = np.array(selected_bands)
meta.update(count=i, dtype=new_tiff_array.dtype)
with rasterio.open(output_tiff, "w", **meta) as dst:
    dst.write(new_tiff_array)

print(f"New TIFF saved: {output_tiff}")


===start reorder bands==
start process the band 0
start process the band 1
start process the band 2
start process the band 3
start process the band 4
start process the band 5
start process the band 6
start process the band 7
start process the band 8
start process the band 9
start process the band 10
start process the band 11
start process the band 12
start process the band 13
start process the band 14
start process the band 15
start process the band 16
start process the band 17
start process the band 18
(3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.39999995

NameError: name 'new_tiff_array' is not defined

In [8]:
###########################################################################
#######################                            ########################
#######################     tiff reorder part 02   ########################
#######################  extract all single band   ########################
#######################                            ########################
###########################################################################
# tiff band reorder
# Load the CSV file containing band selection info
csv_file = "./predictionMap2025Jan/predictionBandsInfo2.csv"
df = pd.read_csv(csv_file)

# Read the TIFF file
tiff_file = "/blue/changzhao/ji.song/Features_Stack_FL/composite_all.tif"

# Prepare a list to store the selected bands
selected_bands = []
feature_names = []

print("===start reorder bands==")
i = 0
np_out_name = "./predictionMap2025Jan/band1_20.npy"
for _, row in df.iterrows():
    # print(row)
    i = i + 1
    num_list = str(row["tiff_order"]).split(",")
    if len(num_list) < 2:
        band_indice = int(row["tiff_order"])
        print(f"start process the band {i} with tiff_order {band_indice}")
        selected_bands.append(band_indice)

print(f"selected_bands length is {len(selected_bands)}")

with rasterio.open(tiff_file) as src:
    print(src.nodatavals)
    meta = src.meta.copy()  # Save metadata
    band_compo = src.read(selected_bands)  # Load all bands as a NumPy array (shape: bands, height, width)

print(band_compo.shape)
# np.save("/blue/changzhao/zhou.tang/SOC/band_all_single.npy",band_compo)

# Update metadata for the new TIFF
meta.update(count=len(selected_bands), dtype=band_compo.dtype)
# Save the new TIFF file
output_tiff = "/blue/changzhao/zhou.tang/SOC/band_all_single.tif"
with rasterio.open(output_tiff, "w", **meta) as dst:
    dst.write(band_compo)

(21537, 25066)
1
New TIFF saved: /blue/changzhao/zhou.tang/SOC/InputBands_NearFire.tif


In [ ]:
###########################################################################
#######################                            ########################
#######################     tiff reorder part 03   ########################
#######################  combine two tiff together ########################
#######################                            ########################
###########################################################################
# combine two tiff together
# Paths to input TIFFs
tiff_43bands_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/band_all_single.tif"
tiff_1band_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/InputBands_NearFire.tif"
output_tiff_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"

print("loading 43 bands tiff")
# Open the 43-band TIFF
with rasterio.open(tiff_43bands_path) as src1:
    bands_43 = src1.read()  # Read all 43 bands
    meta = src1.meta  # Get metadata

print("loading 1 bands tiff")
# Open the 1-band TIFF
with rasterio.open(tiff_1band_path) as src2:
    band_1 = src2.read(1)  # Read the single band

    # Ensure dimensions match
    if bands_43.shape[1:] != band_1.shape:
        raise ValueError("The dimensions of the two TIFFs do not match.")

# Stack the 1-band raster as a new layer
bands_43 = np.vstack([bands_43, band_1[np.newaxis, :, :]])
print("finish stacking")
# Update metadata for 44 bands
meta.update(count=44)

# Write the new 44-band TIFF
with rasterio.open(output_tiff_path, "w", **meta) as dst:
    dst.write(bands_43)

print(f"44-band TIFF saved to {output_tiff_path}")



loading 43 bands tiff
loading 1 bands tiff


In [2]:
# Paths to input TIFFs
tiff_43bands_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/band_all_single.tif"
tiff_1band_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/InputBands_NearFire.tif"
output_tiff_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"

# Open the 43-band TIFF
with rasterio.open(tiff_43bands_path) as src1:
    meta = src1.meta  # Get metadata
    meta.update(count=44)  # Update metadata for 44 bands

    # Open the output file and write the 43 bands one by one
    with rasterio.open(output_tiff_path, "w", **meta) as dst:
        for band in range(1, src1.count + 1):  # Loop through 43 bands
            data = src1.read(band)
            dst.write(data, band)  # Write each band separately
            
        print("finish the 43 band writing")
        # Open the 1-band TIFF and write it as the 44th band
        with rasterio.open(tiff_1band_path) as src2:
            band_44 = src2.read(1)
            dst.write(band_44, 44)  # Write as the 44th band

print(f"44-band TIFF saved to {output_tiff_path}")

finish the 43 band writing
44-band TIFF saved to /blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif


In [ ]:
###########################################################################
#######################                            ########################
#######################       prediction part 01   ########################
#######################                            ########################
###########################################################################


# Load the CSV file that defines the feature names
csv_file = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/predictionBandsInfo4.csv"
df = pd.read_csv(csv_file)

# Load the trained Random Forest model from a joblib file
model_file = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/soc_predictionMap_20250203_order4.joblib"
rf_model = joblib.load(model_file)

# Extract feature names from the CSV
selected_features = df["Feature_name"].tolist()  # Features expected by the model

# Load the reordered TIFF containing the selected bands
# tiff_file = "/blue/changzhao/zhou.tang/SOC/band_all_single.tif"
tiff_file = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"
with rasterio.open(tiff_file) as src:
    meta = src.meta.copy()
    nodata_values = src.nodatavals  # Get per-band NoData values
    band_data = src.read(masked=True)  # Read as masked array

print(f"nodata_values is {nodata_values}")
# Convert to NumPy array and replace extreme NoData values
band_data = np.where(np.isfinite(band_data), band_data, np.nan)  # Replace inf/-inf with NaN
for i in range(band_data.shape[0]):
    if nodata_values[i] is not None and abs(nodata_values[i]) > 100000:
        print(f"Band {i+1}: Replacing extreme NoData value {nodata_values[i]} with NaN.")
        band_data[i] = np.where(band_data[i] == nodata_values[i], np.nan, band_data[i])

band_data = np.clip(band_data, -1e8, 1e8)
print(f"band shape is {band_data.shape}")
# Reshape to (num_pixels, num_bands)
num_bands, height, width = band_data.shape
band_data = band_data.reshape(num_bands, -1).T

# Convert to DataFrame
band_data = pd.DataFrame(band_data, columns=selected_features)
print(f"df_feature shape {band_data.shape}")
# Ensure feature names match model expectations
expected_features = rf_model.feature_names_in_
band_data = band_data[expected_features]  # Select only matching features
band_data.to_csv("/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/df_features_44bands.csv")
print("save the band data after transfer to a csv table")

# Drop rows with NaN or inf
band_data = band_data.replace([np.inf, -np.inf], np.nan)  # Convert inf to NaN
valid_mask = band_data.notna().all(axis=1)  # Mask valid rows
valid_mask.to_csv("/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/valid_mask_44bands.csv")


###########################################################################
#######################                            ########################
#######################       prediction part 02   ########################
#######################                            ########################
###########################################################################

chunk_size = 1000000
num_pixels = output_array_flat.shape[0]
num_chunks = (num_pixels // chunk_size) + 1 

band_data_valid = band_data[valid_mask]  # Keep only valid rows
predictions_flat = np.full(band_data_valid.shape[0], np.nan)

output_array = np.full((21537, 25066), np.nan)
output_array_flat = output_array.flatten()

for i in range(num_chunks):
    # Get start and end indices for the current chunk
    start_idx = i * chunk_size
    end_idx = min(start_idx + chunk_size, num_pixels)
    
    # Extract the current chunk of features and mask
    features_chunk_df = band_data_valid.iloc[start_idx:end_idx,:]
    print(f"start predict from {start_idx} to {end_idx}")

    if features_chunk_df.size > 0:

        # Make predictions on the current chunk
        predictions_chunk = rf_model.predict(features_chunk_df)
        # print(predictions_chunk)
        # Assign predictions to the corresponding positions in the flattened output array
        predictions_flat[start_idx:end_idx] = predictions_chunk

output_array_flat[valid_mask] = predictions_flat
# Reshape the output array back to the original image shape
output_array = output_array_flat.reshape((21537, 25066))
output_array.to_csv("/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.csv")
print("get all prediction")

# Update metadata for single-band output
meta.update(count=1, dtype=output_array.dtype, nodata=np.nan)

# Save the prediction TIFF
output_tiff = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.tif"
with rasterio.open(output_tiff, "w", **meta) as dst:
    dst.write(output_array, 1)

print(f"Prediction TIFF saved: {output_tiff}")

/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


nodata_values is (3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38, 3.3999999521443642e+38

In [3]:
###########################################################################
#######################                            ########################
#######################       prediction part 03   ########################
#######################   Memory efficent solution ########################
###########################################################################
# Input files
tiff_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"
band_info_csv = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/predictionBandsInfo4.csv"  # Contains feature names
model_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/soc_predictionMap_20250203_order4.joblib" # Trained RF model
csv_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.csv"
tiff_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.tif"

# Load the trained Random Forest model
model = joblib.load(model_path)

# Load feature names
band_info = pd.read_csv(band_info_csv)
if len(band_info) != 44:
    raise ValueError("band_info.csv must contain exactly 44 feature names.")
feature_names = band_info["Feature_name"].tolist()

# Open the TIFF file to process in chunks
with rasterio.open(tiff_path) as src:
    height, width = src.height, src.width  # Get raster size
    num_bands = src.count  # Should be 44
    nodata_values = src.nodatavals  # Per-band NoData values
    
    # Define chunk size
    chunk_size = 500  # Number of rows per chunk

    # Create empty array for storing predictions
    prediction_array = np.full((height, width), np.nan, dtype=np.float32)

    # Open CSV output file
    with open(csv_output_path, "w") as f:
        # Write CSV header
        f.write("Row,Col,Prediction\n")
        
        start_time = time.time()

        # Process image in row chunks
        for row_start in range(0, height, chunk_size):
            
            row_end = min(row_start + chunk_size, height)  # Ensure we don't exceed the image bounds
            window = rasterio.windows.Window(0, row_start, width, row_end - row_start)
            # print(f"window is {row_start}, width is {width}, height is {row_end - row_start}")

            # Read chunk (all bands at once)
            data = src.read(window=window).astype(np.float32)  # Shape: (bands, rows, cols)

            # Replace per-band NoData values with NaN
            for i in range(num_bands):
                if nodata_values[i] is not None:
                    data[i][data[i] == nodata_values[i]] = np.nan

            # Replace inf/-inf with NaN
            data[np.isinf(data)] = np.nan

            # Clip values to (-1e8, 1e8)
            np.clip(data, -1e8, 1e8, out=data)

            # Reshape data: Convert (bands, rows, cols) → (rows*cols, bands)
            reshaped_data = np.moveaxis(data, 0, -1).reshape(-1, num_bands)

            # Generate row and column indices
            cols, rows = np.meshgrid(np.arange(width), np.arange(row_start, row_end))
            indices = np.column_stack([rows.ravel(), cols.ravel()])

            # Convert to DataFrame
            df = pd.DataFrame(reshaped_data, columns=feature_names)
            df["Row"] = indices[:, 0]
            df["Col"] = indices[:, 1]

            # Remove rows with NaN before prediction
            valid_rows = df.dropna()
            if not valid_rows.empty:
                # Make predictions
                predictions = model.predict(valid_rows[feature_names])

                # Store predictions in the TIFF output array
                prediction_array[valid_rows["Row"].values, valid_rows["Col"].values] = predictions

                # Save predictions to CSV
                valid_rows["Prediction"] = predictions
                valid_rows[["Row", "Col", "Prediction"]].to_csv(f, index=False, header=False, mode="a")

            print(f"Processed rows {row_start} to {row_end}...")

# Save predictions as a new single-band TIFF
with rasterio.open(
    tiff_output_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,  # Single-band output
    dtype=np.float32,
    crs=src.crs,
    transform=src.transform
) as dst:
    dst.write(prediction_array, 1)

print(f"Processing complete. Predictions saved to {csv_output_path} and {tiff_output_path}")


/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/scratch/local/57279607/ipykernel_1959159/306515268.py:42: RuntimeWarning: overflow encountered in cast
  data = src.read(window=windo

Processed rows 0 to 500...


/scratch/local/57279607/ipykernel_1959159/306515268.py:42: RuntimeWarning: overflow encountered in cast
  data = src.read(window=window).astype(np.float32)  # Shape: (bands, rows, cols)
/scratch/local/57279607/ipykernel_1959159/306515268.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_rows["Prediction"] = predictions


Processed rows 500 to 1000...


/scratch/local/57279607/ipykernel_1959159/306515268.py:42: RuntimeWarning: overflow encountered in cast
  data = src.read(window=window).astype(np.float32)  # Shape: (bands, rows, cols)
/scratch/local/57279607/ipykernel_1959159/306515268.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_rows["Prediction"] = predictions


...

Processed rows 20500 to 21000...


/scratch/local/57279607/ipykernel_1959159/306515268.py:42: RuntimeWarning: overflow encountered in cast
  data = src.read(window=window).astype(np.float32)  # Shape: (bands, rows, cols)
/scratch/local/57279607/ipykernel_1959159/306515268.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_rows["Prediction"] = predictions


Processed rows 21000 to 21500...


/scratch/local/57279607/ipykernel_1959159/306515268.py:42: RuntimeWarning: overflow encountered in cast
  data = src.read(window=window).astype(np.float32)  # Shape: (bands, rows, cols)
/scratch/local/57279607/ipykernel_1959159/306515268.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_rows["Prediction"] = predictions


Processed rows 21500 to 21537...
Processing complete. Predictions saved to /blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.csv and /blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203.tif


In [7]:
# Input files
tiff_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"
band_info_csv = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/predictionBandsInfo4.csv"  # Contains feature names
model_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/soc_predictionMap_20250203_order4.joblib" # Trained RF model
csv_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.csv"
tiff_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.tif"

# Load the trained Random Forest model
model = joblib.load(model_path)

# Load feature names
band_info = pd.read_csv(band_info_csv)
if len(band_info) != 44:
    raise ValueError("band_info.csv must contain exactly 44 feature names.")
feature_names = band_info["Feature_name"].tolist()

# Open the TIFF file to process in chunks
with rasterio.open(tiff_path) as src:
    height, width = src.height, src.width
    num_bands = src.count
    nodata_values = src.nodatavals

    chunk_size = 500  # Number of rows per chunk
    num_chunks = (height // chunk_size) + 1  # Total number of chunks

    # Create empty array for storing predictions
    prediction_array = np.full((height, width), np.nan, dtype=np.float32)

    # Open CSV output file
    with open(csv_output_path, "w") as f:
        f.write("Row,Col,Prediction\n")  # Write CSV header

        # Start timer
        start_time = time.time()

        # Process with a progress bar
        for row_start in tqdm(range(0, height, chunk_size), total=num_chunks, desc="Processing", unit="chunk"):
            row_end = min(row_start + chunk_size, height)
            window = rasterio.windows.Window(0, row_start, width, row_end - row_start)

            # Read chunk (all bands)
            data = src.read(window=window).astype(np.float32)

            # Replace NoData values with NaN
            for i in range(num_bands):
                if nodata_values[i] is not None:
                    data[i][data[i] == nodata_values[i]] = np.nan

            # Handle inf values
            data[np.isinf(data)] = np.nan

            # Clip extreme values
            np.clip(data, -1e10, 1e10, out=data)

            # Reshape data
            reshaped_data = np.moveaxis(data, 0, -1).reshape(-1, num_bands)
            cols, rows = np.meshgrid(np.arange(width), np.arange(row_start, row_end))
            indices = np.column_stack([rows.ravel(), cols.ravel()])

            # Convert to DataFrame
            df = pd.DataFrame(reshaped_data, columns=feature_names)
            df["Row"] = indices[:, 0]
            df["Col"] = indices[:, 1]

            # Drop rows with NaN before prediction
            valid_rows = df.dropna()
            if not valid_rows.empty:
                predictions = model.predict(valid_rows[feature_names])

                # Store predictions in TIFF array
                prediction_array[valid_rows["Row"].values, valid_rows["Col"].values] = predictions

                # Save predictions to CSV
                valid_rows["Prediction"] = predictions
                valid_rows[["Row", "Col", "Prediction"]].to_csv(f, index=False, header=False, mode="a")

        # Stop timer and print total time used
        total_time = time.time() - start_time
        print(f"\nTotal processing time: {total_time:.2f} seconds")

# Save predictions as a single-band TIFF
with rasterio.open(
    tiff_output_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=np.float32,
    crs=src.crs,
    transform=src.transform
) as dst:
    dst.write(prediction_array, 1)

print(f"Predictions saved to {csv_output_path} and {tiff_output_path}")


/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/blue/changzhao/zhou.tang/conda/envs/soc_gpu/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.3.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Processing:   0%|          | 0/44 [00:00<?, ?chunk/s]/scratch/local/57279607/ipykernel_1959159/3949264445.py:42: RuntimeWarning: overf


Total processing time: 2327.81 seconds
Predictions saved to /blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.csv and /blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.tif


In [ ]:
# quantile prediction

# Input files
tiff_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/output_44bands.tif"
band_info_csv = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/predictionBandsInfo4.csv"  # Contains feature names
model_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/soc_predictionMap_20250203_order4.joblib" # Trained RF model
csv_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.csv"
tiff_output_path = "/blue/changzhao/zhou.tang/SOC/predictionMap_44bands_20250203/prediction_44bands_20250203_2.tif"

# Load the trained Random Forest model
model = joblib.load(model_path)

# Load feature names
band_info = pd.read_csv(band_info_csv)
if len(band_info) != 44:
    raise ValueError("band_info.csv must contain exactly 44 feature names.")
feature_names = band_info["Feature_name"].tolist()

# Open the TIFF file to process in chunks
with rasterio.open(tiff_path) as src:
    height, width = src.height, src.width
    num_bands = src.count
    nodata_values = src.nodatavals

    chunk_size = 500  # Number of rows per chunk
    num_chunks = (height // chunk_size) + 1  # Total number of chunks

    # Create empty array for storing predictions
    prediction_array = np.full((height, width), np.nan, dtype=np.float32)

    # Open CSV output file
    with open(csv_output_path, "w") as f:
        f.write("Row,Col,Prediction\n")  # Write CSV header

        # Start timer
        start_time = time.time()

        # Process with a progress bar
        for row_start in tqdm(range(0, height, chunk_size), total=num_chunks, desc="Processing", unit="chunk"):
            row_end = min(row_start + chunk_size, height)
            window = rasterio.windows.Window(0, row_start, width, row_end - row_start)

            # Read chunk (all bands)
            data = src.read(window=window).astype(np.float32)

            # Replace NoData values with NaN
            for i in range(num_bands):
                if nodata_values[i] is not None:
                    data[i][data[i] == nodata_values[i]] = np.nan

            # Handle inf values
            data[np.isinf(data)] = np.nan

            # Clip extreme values
            np.clip(data, -1e10, 1e10, out=data)

            # Reshape data
            reshaped_data = np.moveaxis(data, 0, -1).reshape(-1, num_bands)
            cols, rows = np.meshgrid(np.arange(width), np.arange(row_start, row_end))
            indices = np.column_stack([rows.ravel(), cols.ravel()])

            # Convert to DataFrame
            df = pd.DataFrame(reshaped_data, columns=feature_names)
            df["Row"] = indices[:, 0]
            df["Col"] = indices[:, 1]

            # Drop rows with NaN before prediction
            valid_rows = df.dropna()
            if not valid_rows.empty:
                predictions = model.predict(valid_rows[feature_names])

                # Store predictions in TIFF array
                prediction_array[valid_rows["Row"].values, valid_rows["Col"].values] = predictions

                # Save predictions to CSV
                valid_rows["Prediction"] = predictions
                valid_rows[["Row", "Col", "Prediction"]].to_csv(f, index=False, header=False, mode="a")

        # Stop timer and print total time used
        total_time = time.time() - start_time
        print(f"\nTotal processing time: {total_time:.2f} seconds")

# Save predictions as a single-band TIFF
with rasterio.open(
    tiff_output_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=np.float32,
    crs=src.crs,
    transform=src.transform
) as dst:
    dst.write(prediction_array, 1)

print(f"Predictions saved to {csv_output_path} and {tiff_output_path}")
